# Black–Scholes and Monte Carlo Pricing

This notebook puts module 10 together. The hedging argument of 10/01, applied in continuous time with Itô's lemma (10/04), gives the Black–Scholes
partial differential equation. Its solution is a risk-neutral expectation over the lognormal distribution of 10/03, which can be evaluated in
closed form for a European call and by Monte Carlo simulation for almost anything else. The notebook derives the formula and its Greeks, prices
options by Monte Carlo with error bars and variance reduction, inverts prices into implied volatilities, and shows how a model with jumps produces
a volatility smile that Black–Scholes cannot.

## 0. Math link

- Risk-neutral pricing and the binomial tree: [10/01](./01_binomial_tree_and_risk_neutral_pricing.ipynb).
- Brownian motion and GBM: [10/02](./02_from_random_walk_to_brownian_motion.ipynb), [10/03](./03_geometric_brownian_motion_and_lognormal_prices.ipynb).
- Itô's lemma and the delta-hedged P&L: [10/04](./04_itos_lemma_intuitively.ipynb).
- [Central limit theorem](https://github.com/dnlwrthstr/LearningProbability/blob/main/coursera/04_sampling_and_point_estimation/07_central_limit_theorem_continuous.ipynb)
  and [confidence intervals](https://github.com/dnlwrthstr/LearningProbability/blob/main/coursera/06_confidence_intervals/04_calculating_confidence_intervals.ipynb):
  a Monte Carlo price is a sample mean with a standard error.

## 1. Introduction

A bank sells a one-year call on a Swiss stock and a one-year Asian option on the same stock (paying on the average price over the year). The call has a
closed-form price. The Asian option does not, but both are expectations of a discounted payoff under the risk-neutral GBM, and simulation can
estimate any such expectation with a known error.

After this notebook you can:

- derive the Black–Scholes PDE from a riskless hedge and explain why $\mu$ disappears;
- derive the Black–Scholes formula as a risk-neutral expectation and read $N(d_1)$ and $N(d_2)$;
- compute and interpret the Greeks;
- price options by Monte Carlo with standard errors, and reduce the error with antithetic and control variates;
- compute implied volatilities and recognize a volatility smile.

## 2. Theory

### The Black–Scholes PDE

Let the stock follow GBM, $dS = \mu S\,dt + \sigma S\,dW$, and let $V(t, S)$ be the option value. Hold the option and short $\Delta = V_S$ shares. By Itô's lemma (10/04) the
position changes by

$$
d(V - V_SS) = \big(V_t + \tfrac12\sigma^2S^2V_{SS}\big)dt
$$

with no $dW$ term: it is riskless over $dt$, so it must earn the risk-free rate, $d(V - V_SS) = r(V - V_SS)\,dt$. Rearranging:

$$
V_t + \tfrac12\sigma^2S^2V_{SS} + rSV_S - rV = 0
$$

with the payoff as terminal condition, e.g. $V(T, S) = \max(S - K, 0)$. The expected return $\mu$ does not appear.

### Risk-neutral valuation

The solution of this PDE (Feynman–Kac) is

$$
V(0, S_0) = e^{-rT}E^Q[\text{payoff}(S_T)], \qquad S_T = S_0\exp\big((r - \tfrac12\sigma^2)T + \sigma\sqrt T Z\big),\ Z \sim N(0, 1)
$$

the continuous-time version of the binomial result: under $Q$ the stock drifts at $r$.

### The formula

For a call, split the expectation at the exercise boundary $S_T > K$:

$$
C = S_0N(d_1) - Ke^{-rT}N(d_2), \qquad d_{1,2} = \frac{\ln(S_0/K) + (r \pm \frac12\sigma^2)T}{\sigma\sqrt T}
$$

- $N(d_2) = Q(S_T > K)$: the risk-neutral probability of exercise (10/03 with $\mu = r$); $Ke^{-rT}N(d_2)$ is the present value of paying the strike.
- $S_0N(d_1)$ is the present value of receiving the stock when exercised; $N(d_1)$ is also the call's delta.
- Put by parity: $P = C - S_0 + Ke^{-rT}$.

### Greeks

With $\phi$ the standard normal density:

| Greek | call | meaning |
|---|---|---|
| delta $\partial V/\partial S$ | $N(d_1)$ | shares to hedge |
| gamma $\partial^2V/\partial S^2$ | $\phi(d_1)/(S\sigma\sqrt T)$ | how fast delta changes |
| vega $\partial V/\partial\sigma$ | $S\phi(d_1)\sqrt T$ | sensitivity to volatility |
| theta $\partial V/\partial t$ | $-\frac{S\phi(d_1)\sigma}{2\sqrt T} - rKe^{-rT}N(d_2)$ | time decay |

The PDE says $\theta + \frac12\sigma^2S^2\Gamma + rS\Delta - rV = 0$: theta pays for gamma (10/04).

### Monte Carlo

Draw $N$ independent $Z_i$, compute discounted payoffs $Y_i = e^{-rT}\text{payoff}(S_T^{(i)})$, and estimate $\hat V = \bar Y$. By the central limit theorem,

$$
\operatorname{SE}(\hat V) = \frac{s_Y}{\sqrt N}
$$

The error falls only like $1/\sqrt N$: ten times more accuracy costs a hundred times more paths. Two classical improvements:

- **Antithetic variates**: use each $Z_i$ together with $-Z_i$; the payoffs are negatively correlated, so their average has lower variance.
- **Control variates**: if another payoff $X$ with known mean $E[X]$ is correlated with $Y$, use $\hat V_{CV} = \bar Y - b(\bar X - E[X])$ with $b = \operatorname{Cov}(X, Y)/\operatorname{Var}(X)$. The variance falls by the factor
  $1 - \rho_{XY}^2$.

Path-dependent options (Asian, barrier, lookback) simulate the whole path with the exact lognormal step of 10/03.

### Implied volatility

Given a market price, the **implied volatility** is the $\sigma$ that makes the Black–Scholes price equal to it; since the price increases with $\sigma$ (vega > 0), it is unique and found by a
root search. If Black–Scholes were the true model, all strikes would give the same implied volatility. In practice they do not: the **smile** or **skew** reflects fat
tails and jumps (09/02, 10/02 exercise 3).

## 3. Worked Examples

$S_0 = 100$ (CHF), $K = 100$, $T = 1$, $r = 1\%$, $\sigma = 20\%$.

### Example 1: the call

$d_1 = (0 + 0.01 + 0.02)/0.2 = 0.15$, $d_2 = -0.05$. $N(d_1) = 0.5596$, $N(d_2) = 0.4801$, $Ke^{-rT} = 99.005$.
$C = 55.96 - 99.005\cdot 0.4801 = 55.96 - 47.53 = 8.43$. Put: $8.43 - 100 + 99.005 = 7.44$.

### Example 2: Greeks

Delta 0.560. Gamma $\phi(0.15)/(100\cdot 0.2) = 0.3945/20 = 0.0197$ per CHF. Vega $100\cdot 0.3945 = 39.4$ per unit of volatility, i.e. 0.39 CHF per volatility point.

### Example 3: Monte Carlo accuracy

If the discounted payoff has standard deviation 13.5 CHF, then 10,000 paths give a standard error of 0.135 CHF; a standard error of 0.01 CHF needs $(13.5/0.01)^2 \approx 1.8$ million paths.

### Example 4: risk-neutral vs real probability

$Q(S_1 > 100) = N(d_2) = 48.0\%$. With a real expected return of 7%, $P(S_1 > 100) = N\big((0.07 - 0.02)/0.2\big) = N(0.25) = 59.9\%$. The price uses the first.

## 4. Finance Application

### Where it matters

- **Market quoting.** Option markets quote implied volatilities, not prices; Black–Scholes is the conversion formula even where nobody believes its assumptions.
- **Risk management of option books.** Greeks aggregate exposures across thousands of options; vega and gamma limits are set in these units.
- **Structured products.** Autocallables, Asian and barrier notes sold to private clients are priced by Monte Carlo on (more realistic versions of) GBM.
- **Model risk.** The smile shows where the model fails; pricing exotic options requires a model that fits it (local or stochastic volatility, jumps).

### The usual misreadings

- "$N(d_2)$ is the probability that the option ends in the money." Under the risk-neutral measure only; the real probability depends on $\mu$.
- "Monte Carlo with 10,000 paths is exact." It has a standard error; report it.
- "Implied volatility is a forecast of realized volatility." It is the market price of volatility, including a risk premium; on average it has exceeded realized volatility for equity indices.

## 5. Python Examples

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import brentq

rng = np.random.default_rng(42)

### The formula, the Greeks and the PDE

In [ ]:
def bs(S, K, T, r, sigma, kind="call"):
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    if kind == "call":
        return S * stats.norm.cdf(d1) - K * np.exp(-r * T) * stats.norm.cdf(d2)
    return K * np.exp(-r * T) * stats.norm.cdf(-d2) - S * stats.norm.cdf(-d1)


def bs_greeks(S, K, T, r, sigma):
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    pdf = stats.norm.pdf(d1)
    return {"delta": stats.norm.cdf(d1), "gamma": pdf / (S * sigma * np.sqrt(T)), "vega": S * pdf * np.sqrt(T),
            "theta": -S * pdf * sigma / (2 * np.sqrt(T)) - r * K * np.exp(-r * T) * stats.norm.cdf(d2),
            "N(d2)": stats.norm.cdf(d2)}


def binomial_crr(S, K, T, r, sigma, n):
    dt = T / n
    u = np.exp(sigma * np.sqrt(dt))
    q = (np.exp(r * dt) - 1 / u) / (u - 1 / u)
    k = np.arange(n + 1)
    V = np.maximum(S * u ** (2 * k - n) - K, 0)
    for _ in range(n):
        V = (q * V[1:] + (1 - q) * V[:-1]) * np.exp(-r * dt)
    return V[0]


S0, K, T, r, sigma = 100.0, 100.0, 1.0, 0.01, 0.20
C, P_ = bs(S0, K, T, r, sigma), bs(S0, K, T, r, sigma, "put")
g = bs_greeks(S0, K, T, r, sigma)
print(f"call {C:.4f}, put {P_:.4f}, parity check {C - P_ - (S0 - K * np.exp(-r * T)):.1e}")
print(f"binomial tree (10/01), 2,000 steps: {binomial_crr(S0, K, T, r, sigma, 2_000):.4f}")
print({k: round(v, 4) for k, v in g.items()})

h = 1e-3
V_S = (bs(S0 + h, K, T, r, sigma) - bs(S0 - h, K, T, r, sigma)) / (2 * h)
V_SS = (bs(S0 + h, K, T, r, sigma) - 2 * C + bs(S0 - h, K, T, r, sigma)) / h**2
V_t = (bs(S0, K, T - h, r, sigma) - bs(S0, K, T + h, r, sigma)) / (2 * h)        # dV/dt = -dV/d(time to maturity)
pde = V_t + 0.5 * sigma**2 * S0**2 * V_SS + r * S0 * V_S - r * C
print(f"finite differences: delta {V_S:.4f}, gamma {V_SS:.4f}, theta {V_t:.4f};  PDE residual {pde:.1e}")

The formula gives 8.4333 for the call and 7.4383 for the put; the 2,000-step binomial tree of 10/01 gives 8.4323. Finite differences reproduce delta, gamma and theta,
and plugging them into the PDE leaves a residual at the level of the finite-difference error.

### Monte Carlo for the European call

In [ ]:
def mc_call(S, K, T, r, sigma, N, rng_, method="plain"):
    if method == "antithetic":
        Z = rng_.standard_normal(N // 2)
        Z = np.concatenate([Z, -Z])
    else:
        Z = rng_.standard_normal(N)
    ST = S * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * Z)
    Y = np.exp(-r * T) * np.maximum(ST - K, 0)
    if method == "antithetic":
        pairs = 0.5 * (Y[: N // 2] + Y[N // 2 :])
        return pairs.mean(), pairs.std(ddof=1) / np.sqrt(N // 2)
    if method == "control":
        X = np.exp(-r * T) * ST                                   # discounted stock: known mean S
        b = np.cov(X, Y)[0, 1] / X.var(ddof=1)
        Ycv = Y - b * (X - S)
        return Ycv.mean(), Ycv.std(ddof=1) / np.sqrt(N)
    return Y.mean(), Y.std(ddof=1) / np.sqrt(N)

rows = []
for N in [1_000, 10_000, 100_000, 1_000_000]:
    for m in ["plain", "antithetic", "control"]:
        est, se = mc_call(S0, K, T, r, sigma, N, rng, m)
        rows.append({"N": N, "method": m, "estimate": est, "SE": se, "error vs BS": est - C})
mc_tbl = pd.DataFrame(rows)
mc_tbl.pivot(index="N", columns="method", values=["estimate", "SE"]).round(4)

In [ ]:
cover = []
for _ in range(1_000):
    est, se = mc_call(S0, K, T, r, sigma, 10_000, rng)
    cover.append(abs(est - C) <= 1.96 * se)
print(f"95% Monte Carlo intervals that contain the Black-Scholes price: {np.mean(cover):.1%}")

The plain standard error falls from 0.41 CHF with 1,000 paths to 0.0135 with a million, a factor of $\sqrt{1000} \approx 32$. Antithetic variates cut it by about 22%, the control variate
(the discounted stock price) by more than half. The 95% intervals contain the true price in 95.8% of 1,000 repetitions.

### An Asian option with a control variate

Arithmetic-average call: pays $\max(\bar S - K, 0)$ with $\bar S$ the average of 12 monthly prices. No closed form. Control variate: the European call on the same paths,
whose price is known.

In [ ]:
def asian_mc(S, K, T, r, sigma, N, n_obs, rng_):
    dt = T / n_obs
    Z = rng_.standard_normal((N, n_obs))
    logS = np.log(S) + np.cumsum((r - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * Z, axis=1)
    paths = np.exp(logS)
    Y = np.exp(-r * T) * np.maximum(paths.mean(axis=1) - K, 0)
    X = np.exp(-r * T) * np.maximum(paths[:, -1] - K, 0)
    b = np.cov(X, Y)[0, 1] / X.var(ddof=1)
    Ycv = Y - b * (X - bs(S, K, T, r, sigma))
    return {"plain": (Y.mean(), Y.std(ddof=1) / np.sqrt(N)), "control variate": (Ycv.mean(), Ycv.std(ddof=1) / np.sqrt(N)),
            "corr(Asian, European)": np.corrcoef(X, Y)[0, 1], "b": b}

asian = asian_mc(S0, K, T, r, sigma, 100_000, 12, rng)
print(f"Asian call, plain MC:  {asian['plain'][0]:.4f} ± {1.96 * asian['plain'][1]:.4f}")
print(f"Asian call, control:   {asian['control variate'][0]:.4f} ± {1.96 * asian['control variate'][1]:.4f}")
print(f"correlation with the European payoff {asian['corr(Asian, European)']:.3f}, variance reduction factor "
      f"{(asian['plain'][1] / asian['control variate'][1])**2:.1f}")

The Asian call is worth about 5.11 CHF, much less than the European 8.43: averaging lowers the volatility of the underlying. The European payoff has correlation 0.85 with the
Asian payoff, so using it as a control variate cuts the variance by a factor of 3.7, the same accuracy as 3.7 times as many plain paths.

### Implied volatility and a smile from jumps

Generate "market" prices from a model with jumps (Merton 1976: GBM plus normally distributed log jumps arriving at rate $\lambda$), and invert them strike by
strike with the Black–Scholes formula.

In [ ]:
def implied_vol(price, S, K, T, r, kind="call"):
    f = lambda s: bs(S, K, T, r, s, kind) - price
    return brentq(f, 1e-4, 3.0)

def merton_mc(S, Ks, T, r, sigma, lam, mu_j, sig_j, N, rng_):
    kappa = np.exp(mu_j + 0.5 * sig_j**2) - 1                 # mean jump size, used to keep the risk-neutral drift at r
    Z = rng_.standard_normal(N)
    n_j = rng_.poisson(lam * T, N)
    J = n_j * mu_j + np.sqrt(n_j) * sig_j * rng_.standard_normal(N)
    ST = S * np.exp((r - 0.5 * sigma**2 - lam * kappa) * T + sigma * np.sqrt(T) * Z + J)
    return np.array([np.exp(-r * T) * np.mean(np.maximum(ST - k, 0)) for k in Ks]), np.exp(-r * T) * ST.mean()

strikes = np.arange(70, 135, 5.0)
prices, disc_mean = merton_mc(S0, strikes, 0.5, r, 0.15, 0.5, -0.10, 0.10, 1_000_000, rng)
ivs = [implied_vol(p, S0, k, 0.5, r) for p, k in zip(prices, strikes)]
print(f"martingale check: discounted E[S_T] = {disc_mean:.3f} (S0 = 100)")
pd.Series(ivs, index=strikes.astype(int)).map(lambda v: f"{v:.2%}").rename("implied vol (T = 0.5)")

The jump model has 15% diffusion volatility and occasional downward jumps (on average −10%, one every two years). Its prices, read through Black–Scholes, give
implied volatilities from 21.5% for a strike of 70 to 16.3% for 130: a skew. Low strikes (out-of-the-money puts, via parity) carry the crash risk that a lognormal
model with one volatility cannot represent. Equity index options show the same pattern.

## 6. Visualizations

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
S_grid = np.linspace(60, 140, 200)
for T_, c in [(1.0, "tab:blue"), (0.25, "tab:orange"), (0.02, "tab:green")]:
    axes[0].plot(S_grid, bs(S_grid, K, T_, r, sigma), color=c, label=f"T = {T_}")
    gg = bs_greeks(S_grid, K, T_, r, sigma)
    axes[1].plot(S_grid, gg["delta"], color=c, label=f"delta, T = {T_}")
axes[0].plot(S_grid, np.maximum(S_grid - K, 0), color="black", linewidth=0.8, label="payoff")
axes[0].set_title("Black-Scholes call price (K = 100, sigma 20%, r 1%)")
axes[0].set_xlabel("S (CHF)")
axes[0].legend()
axes[1].set_title("Delta N(d1)")
axes[1].set_xlabel("S (CHF)")
axes[1].legend()
for ax in axes:
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for m, mk in [("plain", "o-"), ("antithetic", "s-"), ("control", "^-")]:
    sub = mc_tbl[mc_tbl["method"] == m]
    ax.loglog(sub["N"], sub["SE"], mk, label=m)
Ns = np.array([1e3, 1e6])
ax.loglog(Ns, mc_tbl[(mc_tbl["method"] == "plain") & (mc_tbl["N"] == 1_000)]["SE"].iloc[0] * np.sqrt(1e3 / Ns), ":", color="grey",
          label="slope -1/2")
ax.set_title("Monte Carlo standard error of the call price")
ax.set_xlabel("number of paths N")
ax.set_ylabel("standard error (CHF)")
ax.legend()
ax.grid(True, alpha=0.3, which="both")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(strikes, ivs, "o-", label="implied vol of jump-model prices")
ax.axhline(0.15, color="grey", linestyle=":", label="diffusion volatility 15%")
total_vol = np.sqrt(0.15**2 + 0.5 * (0.10**2 + 0.10**2))
ax.axhline(total_vol, color="black", linestyle="--", label=f"total vol incl. jumps {total_vol:.1%}")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.set_title("A volatility skew produced by downward jumps (T = 0.5)")
ax.set_xlabel("strike (CHF)")
ax.set_ylabel("Black-Scholes implied volatility")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

## 7. Exercises

### Basic

1. Price a call and a put with $S_0 = 110$, $K = 100$, $T = 0.5$, $r = 2\%$, $\sigma = 25\%$, and compute the call's delta and gamma.
2. The discounted payoff of an option has standard deviation 20 CHF. How many paths are needed for a 95% interval of ±0.02 CHF?
3. A client reads $N(d_2) = 48\%$ on a term sheet and concludes that the call has a 48% chance of paying off. **Interpret** this number correctly, and estimate the real
   probability under a 7% expected return.

### Intermediate

4. Compute the implied volatility of the call of Example 1 if its market price is 9.00 CHF instead of 8.43. Use the vega to approximate the answer without a root search first.
5. Price the call by Monte Carlo under the **real** measure (drift 7%) and discount at $r$. How far off is it, and why is it wrong?

### Advanced

6. Pathwise delta: show that $\partial/\partial S_0$ of $e^{-rT}\max(S_T - K, 0)$ is $e^{-rT}\mathbb{1}_{S_T > K}S_T/S_0$, estimate the delta by Monte Carlo, and compare with $N(d_1)$.
7. Down-and-out call with barrier 85: price it by Monte Carlo with daily and with monthly monitoring (one year). Which is more valuable and why? Relate the difference to 10/02.

## 8. Solutions / Checks

In [ ]:
# 1.
c1, p1 = bs(110, 100, 0.5, 0.02, 0.25), bs(110, 100, 0.5, 0.02, 0.25, "put")
g1 = bs_greeks(110, 100, 0.5, 0.02, 0.25)
print(f"1. call {c1:.4f}, put {p1:.4f}, delta {g1['delta']:.4f}, gamma {g1['gamma']:.4f}")

# 2.
print(f"2. N = (1.96 * 20 / 0.02)^2 = {(1.96 * 20 / 0.02) ** 2:,.0f}")

# 3.
print(f"3. Q(exercise) {g['N(d2)']:.1%}, real P(S_1 > 100) with mu = 7%: {stats.norm.cdf((0.07 - 0.02) / 0.2):.1%}")

# 4.
approx = 0.20 + (9.00 - C) / g["vega"]
print(f"4. vega approximation {approx:.4f}, exact {implied_vol(9.00, S0, K, T, r):.4f}")

In [ ]:
# 5.
Z5 = rng.standard_normal(1_000_000)
ST_real = S0 * np.exp((0.07 - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * Z5)
print(f"5. real-measure 'price' {np.exp(-r * T) * np.maximum(ST_real - K, 0).mean():.4f} vs Black-Scholes {C:.4f}")

# 6.
ST6 = S0 * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * rng.standard_normal(1_000_000))
pw = np.exp(-r * T) * (ST6 > K) * ST6 / S0
print(f"6. pathwise delta {pw.mean():.4f} ± {1.96 * pw.std() / np.sqrt(pw.size):.4f}, N(d1) = {g['delta']:.4f}")

# 7.
def down_and_out(n_obs, N=200_000, barrier=85.0, rng_=rng):
    dt = T / n_obs
    logS = np.log(S0) + np.cumsum((r - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * rng_.standard_normal((N, n_obs)), axis=1)
    alive = logS.min(axis=1) > np.log(barrier)
    Y = np.exp(-r * T) * np.maximum(np.exp(logS[:, -1]) - K, 0) * alive
    return Y.mean(), 1.96 * Y.std() / np.sqrt(N)
for lab, n_obs in [("daily", 252), ("monthly", 12)]:
    v, ci = down_and_out(n_obs)
    print(f"7. {lab:7s} monitoring: {v:.4f} ± {ci:.4f}  (vanilla call {C:.4f})")

**1.** Call 14.12, put 3.13 (parity holds), delta 0.753, gamma 0.016 per CHF.

**2.** $N = (1.96\cdot 20/0.02)^2 = 3.84$ million paths.

**3.** 48% is the probability under the risk-neutral measure, a pricing device in which the stock earns the risk-free rate. The real probability depends on the expected
return: with 7% it is about 60%. Neither number is needed to price the option, but only the real one says how often a client would see a payoff.

**4.** Vega 39.4 per unit of volatility: $0.20 + 0.57/39.4 = 0.2144$; the root search gives almost the same value.

**5.** Discounting the real-world expected payoff at $r$ gives 12.26 CHF instead of 8.43. It ignores that the option is riskier than a bond: the correct
discount rate for real-world expectations would be higher and changes with $S$ and $t$. Risk-neutral pricing avoids having to know it.

**6.** The pathwise estimate matches $N(d_1)$ within its confidence interval.

**7.** The monthly-monitored option is worth more: it knocks out only if the price is below 85 at a month end, so paths that dip below between observations survive.
Daily monitoring is closer to the continuous barrier. The reflection-principle correction of 10/02 (shifting the barrier by $0.5826\sigma\sqrt{\Delta t}$) approximates the gap.

## 9. Conclusion

- A delta-hedged option is riskless over $dt$, which gives the Black–Scholes PDE; $\mu$ drops out.
- Its solution is a risk-neutral expectation; for a call, $C = S_0N(d_1) - Ke^{-rT}N(d_2)$, with $N(d_2)$ the risk-neutral exercise probability and $N(d_1)$ the delta.
- The Greeks measure exposures; theta pays for gamma.
- Monte Carlo prices any payoff with standard error $s/\sqrt N$; antithetic and control variates cut the variance, a correlated control by $1 - \rho^2$.
- Implied volatility inverts the formula; jumps and fat tails produce a smile or skew that constant-volatility Black–Scholes cannot fit.

This closes module 10, the curriculum's last module. To revisit the foundations, start at [00/01 Simple vs log returns](../00_returns_and_compounding/01_simple_vs_log_returns.ipynb).